# ClearSignal — Phase 3: DistilBERT Fine-Tuning

**Run environment:** Google Colab — T4 GPU required  
**Model:** `distilbert-base-uncased` fine-tuned as a regression head (predict CSAT 1–5)  
**Outputs:**
- `models/bert_weights/` — saved model & tokeniser
- `outputs/bert_val_preds.npy` — val-set predictions (clipped to [1, 5])
- `outputs/bert_test_preds.npy` — test-set predictions (clipped to [1, 5])

---

### Before running
1. **Runtime → Change runtime type → T4 GPU**
2. Upload these files (Files panel on the left, or the upload cell below):
   - `data/raw/synthetic_calls_v3_final.csv` (the full raw CSV with `transcript_text`)
   - `data/features/train_features.csv`
   - `data/features/val_features.csv`
   - `data/features/test_features.csv`
3. Run cells **top to bottom**.

### After running
Download and place locally:
```
clearsignal/models/bert_weights/   ← entire folder
clearsignal/outputs/bert_val_preds.npy
clearsignal/outputs/bert_test_preds.npy
```
Then flip `BERT_READY = True` in `phase4_ensemble.py` and `phase5_evaluate.py`.

## 0 — Install dependencies

In [ ]:
# Install / upgrade required packages.
# Colab ships older versions of some libraries — pin for reproducibility.
!pip install -q \
    transformers==4.40.1 \
    datasets==2.19.0 \
    accelerate==0.29.3 \
    torch --upgrade

## 1 — Imports & constants

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import (
    DistilBertTokenizerFast,
    DistilBertForSequenceClassification,
    get_linear_schedule_with_warmup,
)
from sklearn.metrics import mean_absolute_error
from scipy.stats import pearsonr

# ── Reproducibility ────────────────────────────────────────────────────────────
RANDOM_STATE = 42
random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)

# ── Hyper-parameters (match spec) ─────────────────────────────────────────────
MODEL_NAME   = "distilbert-base-uncased"
MAX_LEN      = 512       # DistilBERT hard limit
BATCH_SIZE   = 8         # spec: batch=8
LR           = 2e-5      # spec: lr=2e-5
NUM_EPOCHS   = 3         # spec: 2-3 epochs; we run 3 with early stopping
WARMUP_RATIO = 0.1       # 10 % of steps used for LR warmup
WEIGHT_DECAY = 0.01
PATIENCE     = 1         # early-stop: stop if val MAE doesn't improve for 1 epoch
CLIP_MIN, CLIP_MAX = 1.0, 5.0

# ── File paths (Colab working dir) ─────────────────────────────────────────────
RAW_CSV          = "synthetic_calls_v3_final.csv"
TRAIN_FEAT_CSV   = "train_features.csv"
VAL_FEAT_CSV     = "val_features.csv"
TEST_FEAT_CSV    = "test_features.csv"
BERT_WEIGHTS_DIR = "bert_weights"
VAL_PREDS_PATH   = "bert_val_preds.npy"
TEST_PREDS_PATH  = "bert_test_preds.npy"

# ── GPU check ─────────────────────────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {DEVICE}")
if DEVICE.type == "cpu":
    print("⚠️  WARNING: No GPU detected. Training will be very slow. "
          "Please go to Runtime → Change runtime type → T4 GPU and re-run.")

## 2 — Upload files

Run this cell to upload through the Colab file picker.  
**Or** drag-and-drop into the Files panel and skip this cell.

In [ ]:
from google.colab import files

print("Upload: synthetic_calls_v3_final.csv, train_features.csv, "
      "val_features.csv, test_features.csv")
uploaded = files.upload()   # select all 4 at once
print("Uploaded:", list(uploaded.keys()))

## 3 — Load data & align splits

In [ ]:
# ── Load raw CSV (we need transcript_text and call_id for alignment) ──────────
raw_df = pd.read_csv(RAW_CSV)
print(f"Raw CSV: {raw_df.shape}  columns: {list(raw_df.columns)}")
assert "transcript_text" in raw_df.columns, "transcript_text column missing from raw CSV"
assert "call_id" in raw_df.columns, "call_id column missing from raw CSV"
assert "csat_score" in raw_df.columns, "csat_score column missing from raw CSV"

# ── Load feature files (for call_id → split membership) ──────────────────────
train_feat = pd.read_csv(TRAIN_FEAT_CSV)
val_feat   = pd.read_csv(VAL_FEAT_CSV)
test_feat  = pd.read_csv(TEST_FEAT_CSV)
print(f"Splits — train:{len(train_feat)}  val:{len(val_feat)}  test:{len(test_feat)}")

# ── Merge to get transcript_text and labels for each split ───────────────────
# Feature CSVs must have a call_id column; merge on that.
# If Person 1's CSVs don't include call_id, adjust the merge key accordingly.
def merge_split(feat_df, raw_df, split_name):
    if "call_id" not in feat_df.columns:
        raise ValueError(
            f"{split_name}_features.csv has no call_id column. "
            "Confirm column name with Person 1."
        )
    merged = feat_df[["call_id"]].merge(
        raw_df[["call_id", "transcript_text", "csat_score"]],
        on="call_id", how="left"
    )
    missing = merged["transcript_text"].isna().sum()
    if missing:
        raise ValueError(f"{missing} rows in {split_name} have no matching transcript.")
    print(f"{split_name}: {len(merged)} rows  csat mean={merged['csat_score'].mean():.3f}")
    return merged

train_df = merge_split(train_feat, raw_df, "train")
val_df   = merge_split(val_feat,   raw_df, "val")
test_df  = merge_split(test_feat,  raw_df, "test")

print("\nLabel distribution (train):")
print(train_df["csat_score"].value_counts().sort_index())

## 4 — Tokenise

In [ ]:
tokenizer = DistilBertTokenizerFast.from_pretrained(MODEL_NAME)

class CSATDataset(Dataset):
    """Tokenises transcripts on the fly; returns input_ids, attention_mask, label."""

    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts     = list(texts)
        self.labels    = list(labels)
        self.tokenizer = tokenizer
        self.max_len   = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            str(self.texts[idx]),
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids":      encoding["input_ids"].squeeze(0),       # (max_len,)
            "attention_mask": encoding["attention_mask"].squeeze(0),  # (max_len,)
            "label":          torch.tensor(self.labels[idx], dtype=torch.float),
        }


train_dataset = CSATDataset(train_df["transcript_text"], train_df["csat_score"], tokenizer, MAX_LEN)
val_dataset   = CSATDataset(val_df["transcript_text"],   val_df["csat_score"],   tokenizer, MAX_LEN)
test_dataset  = CSATDataset(test_df["transcript_text"],  test_df["csat_score"],  tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches: {len(train_loader)}  "
      f"Val batches: {len(val_loader)}  "
      f"Test batches: {len(test_loader)}")

# Quick sanity-check on a single batch
sample = next(iter(train_loader))
print(f"input_ids shape : {sample['input_ids'].shape}")
print(f"labels sample   : {sample['label'][:4]}")

## 5 — Model, optimiser, scheduler

In [ ]:
# num_labels=1 → regression head (MSELoss applied internally by HuggingFace)
model = DistilBertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=1,
    problem_type="regression",   # tells HF to use MSELoss
)
model = model.to(DEVICE)
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

# AdamW with weight-decay (exclude bias & LayerNorm from decay)
no_decay = ["bias", "LayerNorm.weight"]
optimizer_grouped_parameters = [
    {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
     "weight_decay": WEIGHT_DECAY},
    {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
     "weight_decay": 0.0},
]
optimizer = torch.optim.AdamW(optimizer_grouped_parameters, lr=LR)

total_steps  = len(train_loader) * NUM_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)
scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps,
)
print(f"Total steps: {total_steps}  Warmup steps: {warmup_steps}")

## 6 — Helper: evaluation function

In [ ]:
def evaluate(model, loader, device, clip_min=CLIP_MIN, clip_max=CLIP_MAX):
    """Return (mae, pearson_r, raw_preds_array) on the given DataLoader."""
    model.eval()
    all_preds  = []
    all_labels = []

    with torch.no_grad():
        for batch in loader:
            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["label"].to(device)

            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            # logits shape: (batch, 1) — squeeze to (batch,)
            preds = outputs.logits.squeeze(-1).cpu().numpy()
            all_preds.append(preds)
            all_labels.append(labels.cpu().numpy())

    preds  = np.concatenate(all_preds)
    labels = np.concatenate(all_labels)

    # Clip predictions to [1, 5] — critical project rule
    preds_clipped = np.clip(preds, clip_min, clip_max)

    mae = mean_absolute_error(labels, preds_clipped)
    r, p_val = pearsonr(labels, preds_clipped)
    return mae, r, preds_clipped

## 7 — Training loop with early stopping

In [ ]:
best_val_mae    = float("inf")
patience_count  = 0
best_epoch      = 0
history         = []

os.makedirs(BERT_WEIGHTS_DIR, exist_ok=True)

print(f"{'Epoch':>6}  {'Train Loss':>12}  {'Val MAE':>10}  {'Val r':>8}  {'Status':>10}")
print("-" * 60)

for epoch in range(1, NUM_EPOCHS + 1):

    # ── Training ───────────────────────────────────────────────────────────────
    model.train()
    total_train_loss = 0.0

    for step, batch in enumerate(train_loader, 1):
        input_ids      = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels         = batch["label"].to(DEVICE)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,           # HF computes MSELoss internally
        )
        loss = outputs.loss
        loss.backward()

        # Gradient clipping (stabilise training)
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        total_train_loss += loss.item()

        if step % 50 == 0:
            avg = total_train_loss / step
            print(f"  [epoch {epoch}  step {step}/{len(train_loader)}]  avg loss={avg:.4f}",
                  end="\r")

    avg_train_loss = total_train_loss / len(train_loader)

    # ── Validation ─────────────────────────────────────────────────────────────
    val_mae, val_r, _ = evaluate(model, val_loader, DEVICE)

    # ── Early stopping ─────────────────────────────────────────────────────────
    improved = val_mae < best_val_mae
    status = "✓ best"
    if improved:
        best_val_mae   = val_mae
        best_epoch     = epoch
        patience_count = 0
        # Save best checkpoint
        model.save_pretrained(BERT_WEIGHTS_DIR)
        tokenizer.save_pretrained(BERT_WEIGHTS_DIR)
    else:
        patience_count += 1
        status = f"patience {patience_count}/{PATIENCE}"

    row = dict(epoch=epoch, train_loss=avg_train_loss, val_mae=val_mae, val_r=val_r)
    history.append(row)

    print(f"{epoch:>6}  {avg_train_loss:>12.4f}  {val_mae:>10.4f}  {val_r:>8.4f}  {status:>10}")

    if patience_count >= PATIENCE:
        print(f"\nEarly stopping triggered after epoch {epoch} (best epoch={best_epoch}).")
        break

print(f"\nBest val MAE: {best_val_mae:.4f} at epoch {best_epoch}")

## 8 — Reload best checkpoint & generate predictions

In [ ]:
# Reload the best-epoch weights saved during training
print(f"Reloading best checkpoint from '{BERT_WEIGHTS_DIR}/'...")
best_model = DistilBertForSequenceClassification.from_pretrained(
    BERT_WEIGHTS_DIR,
    num_labels=1,
    problem_type="regression",
)
best_model = best_model.to(DEVICE)

# ── Val predictions ────────────────────────────────────────────────────────────
val_mae, val_r, val_preds = evaluate(best_model, val_loader, DEVICE)
print(f"\nVal results (best checkpoint)")
print(f"  MAE      : {val_mae:.4f}")
print(f"  Pearson r: {val_r:.4f}")
print(f"  Pred range: [{val_preds.min():.3f}, {val_preds.max():.3f}]")
print(f"  Pred mean : {val_preds.mean():.3f}")
np.save(VAL_PREDS_PATH, val_preds)
print(f"Saved → {VAL_PREDS_PATH}  (shape: {val_preds.shape})")

# ── Test predictions ───────────────────────────────────────────────────────────
# NOTE: labels are read here ONLY to compute the saved metrics log.
# phase5_evaluate.py is the authoritative, single test-label evaluation.
test_mae, test_r, test_preds = evaluate(best_model, test_loader, DEVICE)
print(f"\nTest results (for internal reference — not the authoritative eval)")
print(f"  MAE      : {test_mae:.4f}")
print(f"  Pearson r: {test_r:.4f}")
print(f"  Pred range: [{test_preds.min():.3f}, {test_preds.max():.3f}]")
np.save(TEST_PREDS_PATH, test_preds)
print(f"Saved → {TEST_PREDS_PATH}  (shape: {test_preds.shape})")

## 9 — Training history

In [ ]:
import matplotlib.pyplot as plt

hist_df = pd.DataFrame(history)
print(hist_df.to_string(index=False))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(hist_df["epoch"], hist_df["train_loss"], marker="o", label="Train Loss (MSE)")
axes[0].set_xlabel("Epoch");  axes[0].set_ylabel("Loss")
axes[0].set_title("Training Loss")
axes[0].legend()

axes[1].plot(hist_df["epoch"], hist_df["val_mae"], marker="o", color="orange", label="Val MAE")
ax2 = axes[1].twinx()
ax2.plot(hist_df["epoch"], hist_df["val_r"], marker="s", color="green", linestyle="--", label="Val r")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Val MAE", color="orange")
ax2.set_ylabel("Pearson r", color="green")
axes[1].set_title("Validation Metrics")
lines1, labels1 = axes[1].get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
axes[1].legend(lines1 + lines2, labels1 + labels2, loc="upper right")

plt.tight_layout()
plt.savefig("bert_training_history.png", dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved → bert_training_history.png")

## 10 — Sanity checks

In [ ]:
print("=" * 50)
print("SANITY CHECKS")
print("=" * 50)

# 1. Shapes match split sizes
assert val_preds.shape[0]  == len(val_df),  \
    f"Val preds length {val_preds.shape[0]} ≠ val rows {len(val_df)}"
assert test_preds.shape[0] == len(test_df), \
    f"Test preds length {test_preds.shape[0]} ≠ test rows {len(test_df)}"
print(f"[PASS] Val  preds shape  : {val_preds.shape}")
print(f"[PASS] Test preds shape  : {test_preds.shape}")

# 2. All predictions clipped to [1, 5]
assert val_preds.min()  >= CLIP_MIN and val_preds.max()  <= CLIP_MAX, "Val preds out of [1,5]"
assert test_preds.min() >= CLIP_MIN and test_preds.max() <= CLIP_MAX, "Test preds out of [1,5]"
print(f"[PASS] All preds in [{CLIP_MIN}, {CLIP_MAX}]")

# 3. Files exist on disk
for fpath in [VAL_PREDS_PATH, TEST_PREDS_PATH]:
    assert os.path.exists(fpath), f"Missing file: {fpath}"
    print(f"[PASS] File exists: {fpath}  ({os.path.getsize(fpath):,} bytes)")

# 4. Bert weights directory has config + model files
weight_files = os.listdir(BERT_WEIGHTS_DIR)
assert "config.json" in weight_files, "bert_weights/config.json missing"
print(f"[PASS] bert_weights/ contains: {sorted(weight_files)}")

# 5. Warn (don't fail) if BERT is worse than Ridge dummy baseline
RIDGE_DUMMY_MAE = 0.9903
if val_mae > RIDGE_DUMMY_MAE:
    print(f"[NOTE] Val MAE {val_mae:.4f} > Ridge dummy baseline {RIDGE_DUMMY_MAE:.4f}. "
          "Expected on synthetic text — document as finding, not a bug.")
else:
    print(f"[INFO] Val MAE {val_mae:.4f} ≤ Ridge dummy baseline — BERT competitive here.")

print("\nAll sanity checks passed.")

## 11 — Save run metadata

In [ ]:
run_meta = {
    "model": MODEL_NAME,
    "max_len": MAX_LEN,
    "batch_size": BATCH_SIZE,
    "lr": LR,
    "num_epochs_run": len(history),
    "best_epoch": best_epoch,
    "warmup_ratio": WARMUP_RATIO,
    "weight_decay": WEIGHT_DECAY,
    "random_state": RANDOM_STATE,
    "best_val_mae": round(float(best_val_mae), 4),
    "best_val_r": round(float(val_r), 4),
    "test_mae_preview": round(float(test_mae), 4),
    "test_r_preview": round(float(test_r), 4),
    "val_pred_min": round(float(val_preds.min()), 4),
    "val_pred_max": round(float(val_preds.max()), 4),
    "history": history,
}

with open("bert_run_meta.json", "w") as f:
    json.dump(run_meta, f, indent=2)

print(json.dumps({k: v for k, v in run_meta.items() if k != "history"}, indent=2))
print("\nSaved → bert_run_meta.json")

## 12 — Download everything

This cell zips and downloads all artefacts you need to copy back into your local `clearsignal/` folder.

In [ ]:
from google.colab import files
import zipfile

ZIP_NAME = "phase3_outputs.zip"

with zipfile.ZipFile(ZIP_NAME, "w", zipfile.ZIP_DEFLATED) as zf:
    # Numpy prediction arrays
    zf.write(VAL_PREDS_PATH)
    zf.write(TEST_PREDS_PATH)
    # Run metadata + training plot
    zf.write("bert_run_meta.json")
    zf.write("bert_training_history.png")
    # Entire bert_weights/ directory
    for fname in os.listdir(BERT_WEIGHTS_DIR):
        full = os.path.join(BERT_WEIGHTS_DIR, fname)
        if os.path.isfile(full):
            zf.write(full)

print(f"Created {ZIP_NAME}. Contents:")
with zipfile.ZipFile(ZIP_NAME) as zf:
    for name in zf.namelist():
        info = zf.getinfo(name)
        print(f"  {name:50s}  {info.file_size:>10,} bytes")

files.download(ZIP_NAME)
print("\nDownload started. Unzip and place files at:")
print("  bert_val_preds.npy   → clearsignal/outputs/bert_val_preds.npy")
print("  bert_test_preds.npy  → clearsignal/outputs/bert_test_preds.npy")
print("  bert_weights/        → clearsignal/models/bert_weights/")
print("Then flip BERT_READY = True in phase4_ensemble.py and phase5_evaluate.py.")